# 04. Advanced Analysis — 고급 분석

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/dartlab/blob/master/notebooks/tutorials/04_advanced.ipynb)

주석 세부항목, 유형자산 변동표, 관계기업 분석 등 심화 모듈을 다룬다.

**학습 내용:**
- 주석 세부항목 (23개 키워드)
- 유형자산 변동표
- 관계기업·공동기업 투자
- 여러 종목 비교 분석

In [ ]:
!pip install -q dartlab

In [ ]:
from dartlab import Company

samsung = Company("005930")

## 1. 주석 세부항목

재무제표 주석에서 특정 항목의 상세 테이블을 추출한다.

In [ ]:
inventory = samsung.notesDetail("재고자산")
inventory.tableDf

In [ ]:
eps = samsung.notesDetail("주당이익")
eps.tableDf

In [ ]:
borrowings = samsung.notesDetail("차입금")
borrowings.tableDf

### 연도별 상세 테이블

In [ ]:
for year, periods in inventory.tables.items():
    for p in periods:
        print(f"[{year}] {p.period} (패턴: {p.pattern})")
        for item in p.items[:3]:
            print(f"  {item.name}: {item.values}")
        print()

## 2. 유형자산 변동표

토지, 건물, 기계장치 등 카테고리별 취득·처분·감가상각을 추적한다.

In [ ]:
ta = samsung.tangibleAsset()

print(f"신뢰도: {ta.reliability}")
if ta.warnings:
    print(f"경고: {ta.warnings}")

ta.movementDf

## 3. 관계기업·공동기업

관계기업의 지분율, 장부가, 변동내역을 분석한다.

In [ ]:
aff = samsung.affiliates()

if aff:
    print(f"분석 연도: {aff.nYears}")
    print("\n--- 변동 시계열 ---")
    print(aff.movementDf)

## 4. 여러 종목 비교 분석

여러 종목을 분석해서 비교한다.

In [ ]:
import polars as pl

codes = ["005930", "000660", "035420"]
rows = []

for code in codes:
    c = Company(code)
    result = c.analyze()
    if result:
        rows.append({
            "기업": result.corpName,
            "분석연도": result.nYears,
            "매칭률": f"{result.allRate:.1%}" if result.allRate else "-",
            "변경점": result.nBreakpoints
        })

pl.DataFrame(rows)

### 배당 비교

In [ ]:
dividends = []
for code in codes:
    c = Company(code)
    result = c.dividend()
    if result and result.timeSeries is not None:
        last = result.timeSeries.row(-1, named=True)
        dividends.append({
            "기업": c.corpName,
            "DPS": last.get("dps"),
            "배당수익률": last.get("dividendYield"),
            "배당성향": last.get("payoutRatio")
        })

pl.DataFrame(dividends)

## 5. 함수 직접 호출

Company 클래스를 거치지 않고 모듈 함수를 직접 호출할 수도 있다.

In [ ]:
from dartlab.finance.summary import analyze
from dartlab.finance.dividend import dividend
from dartlab.finance.notesDetail import notesDetail

result = analyze("005930")
print(f"매칭률: {result.allRate:.1%}")

div = dividend("005930")
print(f"연도 수: {div.nYears}")

notes = notesDetail("005930", "재고자산")
print(f"키워드: {notes.keyword}, 연도 수: {notes.nYears}")